# Validasi Top-K (slm metrics)


| Komponen | Model |
|---|---|
| Language Model | `Qwen/Qwen3-0.6B` |
| Embedding Model | `Qwen/Qwen3-Embedding-0.6B` |
| NLI Evaluator | `MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli` |
| Dataset | `squad_v2 / validation` (campur answerable + unanswerable) |
| Prompt | Config 2 — abstain instruction (English, format `Answer/Evidence`) |
| GPU | T4 cukup |


- **Faithfulness (NLI claim-level)** — `compute_faithfulness_nli` (DeBERTa-v3).
- **Abstention F1** — `detect_abstain_nli` zero-shot, TP/FP/FN/TN.
- `best_k = argmax_k  (faithfulness_answerable + abstention_F1) / 2`

## Desain 3 fase
1. **Phase 1** — Embedding model: encode passages → FAISS index → encode queries (retrieve top-10), lalu **dibuang**.
2. **Phase 2** — LM: use Config 2, generate answers for each k=1..10, parse 'Answer:', then **discard**.
3. **Phase 3** — NLI model (DeBERTa-v3-large, ~1.6GB): faithfulness + abstention F1, pilih `best_k`.

## Output (ke `SAVE_DIR`)
`topk_results_pilot.csv`, `topk_results_full.csv`, `topk_summary_pilot.csv`, `topk_summary_full.csv`, `topk_validation_summary.json`, `generations.json`, `passages.json`, `faiss.index`.

## Cell 1 — Install library

In [ ]:
# Versi terbaru agar kompatibel dgn Qwen3 / Gemma-3 / EmbeddingGemma / e5-mistral.
# PENTING: JANGAN downgrade numpy. Colab pakai numpy 2.x; memaksa "numpy<2" akan bikin
# error "numpy.dtype size changed (Expected 96 ... got 88)".
!pip -q install -U transformers sentence-transformers accelerate datasets faiss-cpu scikit-learn
!pip -q install -U "numpy>=2"
print("Library terpasang.")
print(">>> WAJIB: Runtime > Restart session, lalu jalankan MULAI Cell 3")
print(">>> (jangan run ulang cell install ini setelah restart).")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 138.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 97.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
Library terpasang.
>>> WAJIB: Runtime > Restart session, lalu jalankan MULAI Cell 3
>>> (jangan run ulang cell install ini setelah restart).


> 🔁 **RESTART dulu** setelah cell di atas selesai: menu **Runtime → Restart session**. Setelah restart, **lewati** cell install dan mulai dari Cell 2/3.

## Cell 3 — Import, seed, cek GPU

In [ ]:
import random, os, json, time, gc, warnings, re, string
import numpy as np, pandas as pd
import torch, faiss
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    tot  = torch.cuda.get_device_properties(0).total_memory / 1e9
    free = torch.cuda.mem_get_info()[0] / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM total {tot:.1f} GB | bebas {free:.1f} GB")
else:
    print("GPU tidak tersedia — sangat disarankan pakai GPU.")
print("device:", device, "| seed:", SEED)

GPU: Tesla T4 | VRAM total 15.6 GB | bebas 15.5 GB
device: cuda | seed: 42


## Cell 4 — KONFIGURASI pasangan model (satu-satunya bagian yang beda antar notebook)

In [ ]:
# ============ KONFIGURASI PASANGAN MODEL (EDIT DI SINI) ============
PAIR_ID          = "B1_SLM_qwen3-06b"
STRATEGY         = "B"
KATEGORI         = "SLM"
LM_NAME          = "Qwen/Qwen3-0.6B"
EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
EMB_FAMILY       = "qwen3"   # qwen3 | embeddinggemma | e5mistral

# ---- Dataset (mengikuti slm.ipynb) ----
DATASET_NAME = "rajpurkar/squad_v2"
SPLIT        = "validation"

# ---- Parameter eksperimen top-k ----
K_MIN, K_MAX     = 1, 10        # grid search
N_PILOT          = 18           # Tahap 1 (pilot, interleaved ~9 ans + ~9 unans)
N_FULL           = 100          # Tahap 2 (penuh, 50 ans + 50 unans). Naikkan ke 300 bila kuat.
CORPUS_LIMIT     = None         # None = semua context unik; isi angka bila OOM
EMB_BATCH        = 32
GEN_BATCH        = 4
MAX_NEW_TOKENS   = 150          # Config 2 needs space for 'Answer: ... / Evidence: ...'
MAX_INPUT_TOKENS = 2048

SAVE_DIR = f"/content/drive/MyDrive/topk-eval/{PAIR_ID}"

print("PAIR :", PAIR_ID, "| strategi", STRATEGY, "-", KATEGORI)
print("LM   :", LM_NAME)
print("EMBED:", EMBED_MODEL_NAME, "| family:", EMB_FAMILY)
print("GPU disarankan: T4 cukup")

PAIR : B1_SLM_qwen3-06b | strategi B - SLM
LM   : Qwen/Qwen3-0.6B
EMBED: Qwen/Qwen3-Embedding-0.6B | family: qwen3
GPU disarankan: T4 cukup


## Cell 5 — Mount Drive, load dataset (squad_v2), bangun passages + set evaluasi campur

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(SAVE_DIR, exist_ok=True)

from datasets import load_dataset
ds = load_dataset(DATASET_NAME, split=SPLIT).remove_columns(["title"])

def clean_minimal(t):
    t = t.replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", t).strip()

contexts_clean = [clean_minimal(c) for c in ds["context"]]
passages = list(dict.fromkeys(contexts_clean))
if CORPUS_LIMIT is not None:
    passages = passages[:CORPUS_LIMIT]
print("QA pairs:", len(ds), "| passages unik:", len(passages))

# Set evaluasi CAMPUR (answerable + unanswerable) -> abstention F1 butuh dua-duanya.
# Stratified: 50/50. Diinterleave biar pilot (N_PILOT pertama) juga seimbang.
ans_idx   = [i for i in range(len(ds)) if len(ds["answers"][i]["text"]) > 0]
unans_idx = [i for i in range(len(ds)) if len(ds["answers"][i]["text"]) == 0]
print(f"squad_v2 valid: answerable={len(ans_idx)} | unanswerable={len(unans_idx)}")

rng = random.Random(SEED)
rng.shuffle(ans_idx); rng.shuffle(unans_idx)
n_each = N_FULL // 2
chosen_ans   = ans_idx[:n_each]
chosen_unans = unans_idx[:n_each]
mixed = []
for a, u in zip(chosen_ans, chosen_unans):
    mixed.extend([a, u])             # interleave: HasAns, NoAns, HasAns, NoAns, ...

rows = []
for i in mixed:
    texts = ds["answers"][i]["text"]
    rows.append({
        "qi": i,
        "question": ds["question"][i],
        "gold_answer": texts[0] if len(texts) > 0 else "",
        "gold_is_unanswerable": len(texts) == 0,
    })
df_eval = pd.DataFrame(rows).reset_index(drop=True)
df_pilot = df_eval.iloc[:N_PILOT].reset_index(drop=True)
print("Eval penuh:", len(df_eval),
      "| ans:", (~df_eval["gold_is_unanswerable"]).sum(),
      "| unans:", df_eval["gold_is_unanswerable"].sum())
print("Pilot   :", len(df_pilot),
      "| ans:", (~df_pilot["gold_is_unanswerable"]).sum(),
      "| unans:", df_pilot["gold_is_unanswerable"].sum())
df_eval.head(4)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
QA pairs: 11873 | passages unik: 1204
squad_v2 valid: answerable=5928 | unanswerable=5945
Eval penuh: 100 | ans: 50 | unans: 50
Pilot   : 18 | ans: 9 | unans: 9


,qi,question,gold_answer,gold_is_unanswerable
0,10452,Who did the geographic scholars work for?,colonizing empires,False
1,5838,Who manages construction projects and assumes ...,,True
2,7080,What are both Branko Milanovic and Joseph Stig...,Economist,False
3,9519,The Second World War caused what to be shelved?,,True


## Cell 6 — PHASE 1: embed passages, bangun FAISS index, embed & retrieve queries

In [ ]:
# ===== PHASE 1: EMBED PASSAGES -> INDEX -> EMBED QUERIES (top-K_MAX) =====
# Hanya EMBEDDING model di VRAM. Setelah selesai, embedder dibuang.
from sentence_transformers import SentenceTransformer

def make_embedder():
    kw = {}
    if device == "cuda":
        kw = {"model_kwargs": {"torch_dtype": torch.float16}}
    m = SentenceTransformer(EMBED_MODEL_NAME, device=device,
                            trust_remote_code=True, **kw)
    try: m.max_seq_length = 1024
    except Exception: pass
    return m

def encode_passages(m, texts):
    if EMB_FAMILY == "embeddinggemma":
        return m.encode(texts, prompt="title: none | text: ",
                        normalize_embeddings=True, batch_size=EMB_BATCH,
                        convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
    return m.encode(texts, normalize_embeddings=True, batch_size=EMB_BATCH,
                    convert_to_numpy=True, show_progress_bar=True).astype(np.float32)

def encode_queries(m, texts):
    if EMB_FAMILY == "qwen3":
        instr = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: "
        try:
            return m.encode(texts, prompt_name="query", normalize_embeddings=True,
                            batch_size=EMB_BATCH, convert_to_numpy=True).astype(np.float32)
        except Exception:
            return m.encode([instr + t for t in texts], normalize_embeddings=True,
                            batch_size=EMB_BATCH, convert_to_numpy=True).astype(np.float32)
    if EMB_FAMILY == "embeddinggemma":
        return m.encode(texts, prompt="task: search result | query: ",
                        normalize_embeddings=True, batch_size=EMB_BATCH,
                        convert_to_numpy=True).astype(np.float32)
    if EMB_FAMILY == "e5mistral":
        instr = "Instruct: Given a question, retrieve passages that answer the question\nQuery: "
        return m.encode([instr + t for t in texts], normalize_embeddings=True,
                        batch_size=EMB_BATCH, convert_to_numpy=True).astype(np.float32)
    return m.encode(texts, normalize_embeddings=True, batch_size=EMB_BATCH,
                    convert_to_numpy=True).astype(np.float32)

embedder = make_embedder()
print("Embedding passages...")
p_emb = encode_passages(embedder, passages)
dim   = p_emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(p_emb)
print("index dim:", dim, "| ntotal:", index.ntotal)

print("Embedding queries & retrieval top-K_MAX...")
q_emb = encode_queries(embedder, df_eval["question"].tolist())
D, I  = index.search(np.ascontiguousarray(q_emb.astype(np.float32)), K_MAX)
retrieved_idx   = I.tolist()
retrieved_score = D.tolist()

np.save(os.path.join(SAVE_DIR, "p_emb.npy"), p_emb)
faiss.write_index(index, os.path.join(SAVE_DIR, "faiss.index"))
json.dump(passages, open(os.path.join(SAVE_DIR, "passages.json"), "w"))
json.dump({"idx": retrieved_idx, "score": retrieved_score},
          open(os.path.join(SAVE_DIR, f"retrieval_top{K_MAX}.json"), "w"))

del embedder, p_emb
gc.collect(); torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"VRAM bebas setelah Phase 1: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
print("PHASE 1 selesai — embedder dibuang.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Embedding passages...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

index dim: 1024 | ntotal: 1204
Embedding queries & retrieval top-K_MAX...
VRAM bebas setelah Phase 1: 15.5 GB
PHASE 1 selesai — embedder dibuang.


## Cell 7 — Helper Config 2 (abstain instruction + parser output, matching slm.ipynb)

In [ ]:
# ============================================================
# Helper Config 2 (abstain instruction) — TAKEN EXACTLY from slm.ipynb
# ============================================================
SEP = "\n\n---\n\n"

INSTRUCTION_CONFIG2 = """INSTRUCTION:
Answer the question ONLY based on the information in the CONTEXT. Do not guess or add facts not present in the CONTEXT.

If the CONTEXT does not contain the information needed to answer the question:
- Politely and specifically acknowledge the limitation (mention the aspect of the question that cannot be answered from the CONTEXT).
- Offer one relevant form of further assistance (clarify the question, or a related topic you can help with).
- Avoid generic or template responses.

OUTPUT FORMAT (REQUIRED):
Answer: [your answer based on the CONTEXT, or a polite and contextual abstain statement]
Evidence: [the part of the CONTEXT that supports the answer, or "none" if abstaining]
"""

def parse_output_soft(text, expect_label=False):
    """Extract 'Answer: ...' from Config 2 output. Fallback to raw if parse fails."""
    t = (text or "").strip()
    label = None
    answer_text = t
    evidence_text = ""
    if expect_label:
        m = re.search(r"(?im)^\s*label\s*:\s*(SUPPORTED|UNSUPPORTED)\b", t)
        if m: label = m.group(1).upper()
    m = re.search(r"(?ims)^\s*answer\s*:\s*(.+?)(?=\n\s*evidence\s*:|\Z)", t)
    if m: answer_text = m.group(1).strip()
    m = re.search(r"(?ims)^\s*evidence\s*:\s*(.+?)\Z", t)
    if m: evidence_text = m.group(1).strip()
    return label, answer_text, evidence_text, t

GEN_PARAMS = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                  temperature=0.0, top_p=1.0)
print("Helper Config 2 siap.")

Helper Config 2 siap.


## Cell 8 — PHASE 2: load LM, generate with Config 2 for k=1..10, parse 'Answer:'

In [ ]:
# ===== PHASE 2: GENERATE answers (Config 2) for k=K_MIN..K_MAX =====
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_lm():
    tok = AutoTokenizer.from_pretrained(LM_NAME, use_fast=True, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"; tok.truncation_side = "left"
    dt = torch.float16 if torch.cuda.is_available() else torch.float32
    try:
        mdl = AutoModelForCausalLM.from_pretrained(
            LM_NAME, device_map="auto", torch_dtype=dt, trust_remote_code=True)
    except Exception as e:
        print("AutoModelForCausalLM gagal -> coba ImageTextToText (gemma-3):", repr(e)[:120])
        from transformers import AutoModelForImageTextToText
        mdl = AutoModelForImageTextToText.from_pretrained(
            LM_NAME, device_map="auto", torch_dtype=dt, trust_remote_code=True)
    mdl.config.pad_token_id = tok.pad_token_id
    mdl.eval()
    return tok, mdl

def build_prompt_config2(question, hits_passages, tok):
    """Exact replica of build_prompt_config2 in slm.ipynb (CONTEXT 1..k, English format)."""
    context_text = SEP.join([f"CONTEXT {i+1}:\n{p}" for i, p in enumerate(hits_passages)])
    user_content = f"{INSTRUCTION_CONFIG2}\n\n{context_text}\n\nQUESTION:\n{question}\n"
    messages = [{"role": "user", "content": user_content}]
    try:
        return tok.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_batch(prompts, tok, mdl):
    enc = tok(prompts, return_tensors="pt", padding=True, truncation=True,
              max_length=MAX_INPUT_TOKENS).to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**enc, pad_token_id=tok.pad_token_id,
                           eos_token_id=tok.eos_token_id, **GEN_PARAMS)
    gen = out[:, enc["input_ids"].shape[1]:]
    return [t.strip() for t in tok.batch_decode(gen, skip_special_tokens=True)]

tok, mdl = load_lm()
print("LM loaded:", LM_NAME)

raw_outputs = {k: [None]*len(df_eval) for k in range(K_MIN, K_MAX+1)}
answers     = {k: [None]*len(df_eval) for k in range(K_MIN, K_MAX+1)}
ctx_per_k   = {k: [None]*len(df_eval) for k in range(K_MIN, K_MAX+1)}

for k in range(K_MIN, K_MAX+1):
    t0 = time.time(); prompts = []
    for qi in range(len(df_eval)):
        idxs = retrieved_idx[qi][:k]
        ctxs = [passages[j] for j in idxs if 0 <= j < len(passages)]
        ctx_per_k[k][qi] = ctxs
        prompts.append(build_prompt_config2(df_eval["question"][qi], ctxs, tok))
    outs = []
    for i in range(0, len(prompts), GEN_BATCH):
        outs.extend(generate_batch(prompts[i:i+GEN_BATCH], tok, mdl))
    # parse "Answer:" for NLI faithfulness & abstain detection
    parsed = [parse_output_soft(o)[1] for o in outs]
    raw_outputs[k] = outs
    answers[k]     = parsed
    print(f"k={k:>2} | {time.time()-t0:6.1f}s | sample parsed: {parsed[0][:90]!r}")

json.dump({str(k): {"raw": raw_outputs[k], "parsed": answers[k]} for k in answers},
          open(os.path.join(SAVE_DIR, "generations.json"), "w"), ensure_ascii=False)

del mdl, tok
gc.collect(); torch.cuda.empty_cache()
if torch.cuda.is_available():
    print(f"VRAM bebas setelah Phase 2: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")
print("PHASE 2 selesai — LM dibuang.")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LM loaded: Qwen/Qwen3-0.6B
k= 1 |   94.9s | sample parsed: '[Politik geografis]'
k= 2 |  106.9s | sample parsed: '[The geographic scholars worked for the European empire.]'
k= 3 |  112.1s | sample parsed: '[geografis scholars]'
k= 4 |  111.4s | sample parsed: '[geografis scholars]'
k= 5 |   97.1s | sample parsed: '[geografis scholars]'
k= 6 |  100.6s | sample parsed: '[geographic scholars worked for imperial powers]'
k= 7 |  102.4s | sample parsed: '[geografis scholars]'
k= 8 |  122.9s | sample parsed: '[Geographic scholars worked for imperial powers and colonial empires.]'
k= 9 |  136.7s | sample parsed: '[Geographic scholars worked for the American Empire, as per KONTEKS 3 and 6.]'
k=10 |  158.5s | sample parsed: '[Geographic scholars worked for the European empires and their colonial powers.]'
VRAM bebas setelah Phase 2: 15.5 GB
PHASE 2 selesai — LM dibuang.


## Cell 9 — PHASE 3: NLI faithfulness + abstention F1 → keputusan top-k

In [ ]:
# ===== PHASE 3: NLI -> FAITHFULNESS + ABSTENTION F1 =====
# Taken exactly from slm.ipynb (cells 91, 93, 95, 99).
from transformers import AutoTokenizer, AutoModelForSequenceClassification

NLI_MODEL_NAME = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
print(f"Loading NLI model: {NLI_MODEL_NAME}")
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
nli_model.eval()
NLI_LABELS = ["entailment", "neutral", "contradiction"]
print("NLI ready on:", device)

def split_into_claims(text):
    if not text or not text.strip(): return []
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s.strip() for s in sents if s.strip() and len(s.strip()) > 3]

@torch.no_grad()
def nli_predict(premise, hypothesis):
    inp = nli_tokenizer(premise, hypothesis, truncation=True, max_length=512,
                        return_tensors="pt").to(device)
    logits = nli_model(**inp).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().float().numpy()
    return {"entailment": float(probs[0]), "neutral": float(probs[1]),
            "contradiction": float(probs[2])}

def compute_faithfulness_nli(context, answer):
    if not answer or not answer.strip(): return 0.0
    claims = split_into_claims(answer)
    if not claims: return 0.0
    ctx_cap = context[:3000] if len(context) > 3000 else context
    supported = 0
    for cl in claims:
        p = nli_predict(ctx_cap, cl)
        if p["entailment"] > 0.5 and p["entailment"] > p["contradiction"]:
            supported += 1
    return supported / len(claims)

ABSTAIN_HYPOTHESIS = "This text is a refusal or acknowledgment that the question cannot be answered from the given context."
ANSWER_HYPOTHESIS  = "This text provides a substantive factual answer to a question."

def detect_abstain_nli(text):
    if not text or not text.strip(): return "ABSTAIN"
    t = text[:1500]
    a = nli_predict(t, ABSTAIN_HYPOTHESIS)["entailment"]
    b = nli_predict(t, ANSWER_HYPOTHESIS )["entailment"]
    return "ABSTAIN" if a > b else "ANSWER"

def score_k(k, rows_idx, label_pref=""):
    per = []
    for qi in rows_idx:
        ctx_join = "\n\n".join(ctx_per_k[k][qi] or [])
        ans = answers[k][qi]
        gold_unans = bool(df_eval["gold_is_unanswerable"][qi])
        faith = compute_faithfulness_nli(ctx_join, ans)
        pred_label = detect_abstain_nli(ans)
        gt_label = "ABSTAIN" if gold_unans else "ANSWER"
        per.append({
            "k": k, "qi": int(df_eval["qi"][qi]),
            "gold_is_unanswerable": gold_unans,
            "gt_label": gt_label, "pred_label": pred_label,
            "abstention_correct": int(pred_label == gt_label),
            "faithfulness": faith,
            "answer": ans, "gold_answer": df_eval["gold_answer"][qi],
        })
    return per

def aggregate(per_samples):
    df = pd.DataFrame(per_samples)
    rows = []
    for k, g in df.groupby("k"):
        ans_mask = ~g["gold_is_unanswerable"]
        faith_ans = g.loc[ans_mask, "faithfulness"].mean() if ans_mask.any() else 0.0
        tp = int(((g["gt_label"]=="ABSTAIN") & (g["pred_label"]=="ABSTAIN")).sum())
        fp = int(((g["gt_label"]=="ANSWER")  & (g["pred_label"]=="ABSTAIN")).sum())
        fn = int(((g["gt_label"]=="ABSTAIN") & (g["pred_label"]=="ANSWER")).sum())
        tn = int(((g["gt_label"]=="ANSWER")  & (g["pred_label"]=="ANSWER")).sum())
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        rows.append({"k": int(k),
                     "faithfulness_answerable": round(faith_ans, 4),
                     "abstention_precision": round(prec, 4),
                     "abstention_recall":    round(rec, 4),
                     "abstention_f1":        round(f1, 4),
                     "tp": tp, "fp": fp, "fn": fn, "tn": tn,
                     "decision_score": round((faith_ans + f1) / 2, 4)})
    summ = pd.DataFrame(rows).set_index("k").sort_index()
    return df, summ

def run_stage(rows_idx, label):
    print(f"\nScoring {label} ({len(rows_idx)} sampel x {K_MAX-K_MIN+1} k) ...")
    all_per = []
    for k in range(K_MIN, K_MAX+1):
        t0 = time.time()
        all_per.extend(score_k(k, rows_idx))
        print(f"  k={k:>2} | {time.time()-t0:5.1f}s")
    df_per, summ = aggregate(all_per)
    best = int(summ["decision_score"].idxmax())
    print(summ.to_string())
    for k, row in summ.iterrows():
        mark = "  <- TERBAIK" if k == best else ""
        print(f"  k={k:>2}: faith_ans={row['faithfulness_answerable']:.3f} | "
              f"abst_F1={row['abstention_f1']:.3f} | "
              f"mean={row['decision_score']:.3f}{mark}")
    print(f"KEPUTUSAN ({label}): k={best}")
    return df_per, summ, best

pilot_idx = list(range(min(N_PILOT, len(df_eval))))
full_idx  = list(range(len(df_eval)))
df18,   summ18,   best18   = run_stage(pilot_idx, "Tahap 1 - pilot")
dffull, summfull, bestfull = run_stage(full_idx,  "Tahap 2 - penuh")

if best18 == bestfull:
    print(f"\nKedua tahap SEPAKAT: k={bestfull}")
else:
    print(f"\nTahap berbeda (pilot k={best18}, penuh k={bestfull}) -> pakai penuh: k={bestfull}")

del nli_model, nli_tokenizer
gc.collect(); torch.cuda.empty_cache()

Loading NLI model: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli


config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

NLI ready on: cuda

Scoring Tahap 1 - pilot (18 sampel x 10 k) ...
  k= 1 |   4.6s
  k= 2 |   2.8s
  k= 3 |   2.7s
  k= 4 |   3.0s
  k= 5 |   3.1s
  k= 6 |   3.1s
  k= 7 |   2.7s
  k= 8 |   3.6s
  k= 9 |   3.8s
  k=10 |   4.1s
    faithfulness_answerable  abstention_precision  abstention_recall  abstention_f1  tp  fp  fn  tn  decision_score
k                                                                                                                  
1                    0.6667                0.6000             0.3333         0.4286   3   2   6   7          0.5476
2                    0.6111                0.6000             0.3333         0.4286   3   2   6   7          0.5198
3                    0.7222                0.5000             0.2222         0.3077   2   2   7   7          0.5150
4                    0.6667                0.5000             0.3333         0.4000   3   3   6   6          0.5333
5                    0.3333                0.5000             0.2222         

## Cell 10 — Simpan hasil & keputusan final

In [ ]:
FINAL_K = bestfull
df18.to_csv(os.path.join(SAVE_DIR,  "topk_results_pilot.csv"), index=False)
dffull.to_csv(os.path.join(SAVE_DIR, "topk_results_full.csv"),  index=False)
summ18.to_csv(os.path.join(SAVE_DIR, "topk_summary_pilot.csv"))
summfull.to_csv(os.path.join(SAVE_DIR, "topk_summary_full.csv"))

summary = {
    "pair_id": PAIR_ID, "strategy": STRATEGY, "kategori": KATEGORI,
    "lm": LM_NAME, "embed_model": EMBED_MODEL_NAME,
    "nli_model": "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
    "dataset": f"{DATASET_NAME}/{SPLIT}",
    "prompt_config": "Config 2 (abstain instruction English, format Answer/Evidence)",
    "eval_set": "mixed answerable+unanswerable (50/50 stratified interleaved)",
    "decision_basis": "mean(faithfulness_answerable, abstention_F1) tertinggi",
    "final_k": FINAL_K, "best_k_pilot": best18, "best_k_full": bestfull,
    "n_pilot": len(pilot_idx), "n_full": len(full_idx),
    "k_grid": list(range(K_MIN, K_MAX+1)),
    "scores_pilot": json.loads(summ18.reset_index().to_json(orient="records")),
    "scores_full":  json.loads(summfull.reset_index().to_json(orient="records")),
}
json.dump(summary, open(os.path.join(SAVE_DIR, "topk_validation_summary.json"), "w"), indent=2)

print("=" * 64)
print(f"PASANGAN  : {PAIR_ID}")
print(f"  LM      : {LM_NAME}")
print(f"  EMBED   : {EMBED_MODEL_NAME}")
print(f"  PROMPT  : Config 2 (abstain)")
print(f"  DATASET : {DATASET_NAME}/{SPLIT}  (mixed ans+unans, N={len(df_eval)})")
print(f"  METRIK  : faithfulness_answerable + abstention_F1 (DeBERTa-v3 NLI)")
print(f"  TOP-K FINAL = {FINAL_K}  (pilot k={best18}, penuh k={bestfull})")
print(f"  Tersimpan: {SAVE_DIR}")
print("=" * 64)

PASANGAN  : B1_SLM_qwen3-06b
  LM      : Qwen/Qwen3-0.6B
  EMBED   : Qwen/Qwen3-Embedding-0.6B
  PROMPT  : Config 2 (abstain)
  DATASET : rajpurkar/squad_v2/validation  (mixed ans+unans, N=100)
  METRIK  : faithfulness_answerable + abstention_F1 (DeBERTa-v3 NLI)
  TOP-K FINAL = 2  (pilot k=1, penuh k=2)
  Tersimpan: /content/drive/MyDrive/topk-eval/B1_SLM_qwen3-06b
